## Setup + build metadata + same 70/15/15 split (by writer)

In [18]:
import os, glob, re
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

ROOT_TXT = "/Users/jessica/Documents/WINTER 2026/MATH 156/final_project/assamese_data/Online Handwritten Assamese Characters Dataset"  # <-- change if needed

In [19]:
def build_txt_metadata(root=ROOT_TXT):
    txt_paths = sorted(glob.glob(os.path.join(root, "W*", "*.txt")))
    if len(txt_paths) == 0:
        raise ValueError(f"No .txt files found under {root}/W*/.txt")
    rows = []
    for p in txt_paths:
        writer_id = os.path.basename(os.path.dirname(p))  # W1, W2, ...
        rows.append({"writer_id": writer_id, "txt_path": p})
    return pd.DataFrame(rows)

def stratified_split(df: pd.DataFrame, label_col="writer_id", seed=SEED):
    y = df[label_col].values
    # 1) test = 15%
    df_trainval, df_test = train_test_split(
        df, test_size=0.15, random_state=seed, stratify=y
    )
    # 2) val = 15% overall => val fraction of trainval = 0.15/0.85
    y_trainval = df_trainval[label_col].values
    df_train, df_val = train_test_split(
        df_trainval, test_size=(0.15/0.85), random_state=seed, stratify=y_trainval
    )
    return (
        df_train.reset_index(drop=True),
        df_val.reset_index(drop=True),
        df_test.reset_index(drop=True),
    )

df = build_txt_metadata(ROOT_TXT)
df_train, df_val, df_test = stratified_split(df, label_col="writer_id", seed=SEED)

le = LabelEncoder()
le.fit(df["writer_id"].astype(str))
num_classes = len(le.classes_)

y_train = le.transform(df_train["writer_id"].astype(str))
y_val   = le.transform(df_val["writer_id"].astype(str))
y_test  = le.transform(df_test["writer_id"].astype(str))

print("Total txt samples:", len(df))
print("Num writers:", num_classes)
print("Split sizes:", len(df_train), len(df_val), len(df_test))

Total txt samples: 8030
Num writers: 45
Split sizes: 5620 1205 1205


## Parse .txt → fixed-length time series (T, C)

In [20]:
T = 300                 # sequence length (pad/trim)
USE_PEN_STATE = True    # <-- I recommend True for LSTM (x,y,pen)
X_MAX = 4392.0
Y_MAX = 4868.0

def parse_assamese_txt(path):
    writer_id = os.path.basename(os.path.dirname(path))
    char_name = None
    pts = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("CHARACTER_NAME:"):
                char_name = line.split(":", 1)[1].strip()
                continue

            # skip tokens/headers
            if line.startswith(("PEN_DOWN", "PEN_UP", "END_CHARACTER", "STROKE_COUNT", "X", "Y")):
                continue

            parts = re.split(r"\s+", line)
            if len(parts) >= 4 and parts[0].isdigit() and parts[1].isdigit():
                x = float(parts[0])
                y = float(parts[1])
                pen = float(parts[2])  # 1 for recorded points, 0 for PEN_UP rows (often)
                pts.append([x, y, pen])

    if char_name is None:
        char_name = "UNKNOWN"

    pts = np.array(pts, dtype=np.float32)
    return writer_id, char_name, pts

def to_fixed_length(points_xy_pen, T=300, use_pen=True):
    if points_xy_pen.size == 0:
        C = 6 if use_pen else 5
        return np.zeros((T, C), dtype=np.float32)

    x = points_xy_pen[:, 0]
    y = points_xy_pen[:, 1]
    pen = points_xy_pen[:, 2]

    # Normalize globally (consistent across writers)
    x = x / X_MAX
    y = y / Y_MAX

    # Derivatives
    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])
    speed = np.sqrt(dx**2 + dy**2)

    if use_pen:
        seq = np.stack([x, y, dx, dy, speed, pen], axis=1)
    else:
        seq = np.stack([x, y, dx, dy, speed], axis=1)

    # Trim or pad
    if len(seq) >= T:
        seq = seq[:T]
    else:
        pad = np.zeros((T - len(seq), seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])

    return seq.astype(np.float32)

## tf.data pipeline: .txt → (T,C) tensors

In [23]:
p0 = df_train["txt_path"].values[0]
seq0 = py_load_txt_to_seq(p0.encode("utf-8") if isinstance(p0, str) else p0)
print(seq0.shape)   # should print (300, ?)
print(seq0[:2])     # quick peek

(300, 6)
[[0.41575593 0.22288415 0.         0.         0.         1.        ]
 [0.42167577 0.22288415 0.00591984 0.         0.00591984 1.        ]]


In [24]:
C = 6

def py_load_txt_to_seq(txt_path_bytes):
    path = txt_path_bytes.decode("utf-8")
    _, _, pts = parse_assamese_txt(path)
    seq = to_fixed_length(pts, T=T, use_pen=USE_PEN_STATE)
    return seq  # (T,C)

def tf_load(txt_path, label):
    seq = tf.numpy_function(py_load_txt_to_seq, [txt_path], Tout=tf.float32)
    seq.set_shape((T, C))  # critical for Keras
    return seq, label

def make_seq_dataset(df_part, y_part, batch_size=64, training=False):
    paths = df_part["txt_path"].values.astype(str)
    labels = y_part.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(len(paths), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
train_ds = make_seq_dataset(df_train, y_train, batch_size=BATCH_SIZE, training=True)
val_ds   = make_seq_dataset(df_val,   y_val,   batch_size=BATCH_SIZE, training=False)
test_ds  = make_seq_dataset(df_test,  y_test,  batch_size=BATCH_SIZE, training=False)

xb, yb = next(iter(train_ds))
print("Batch X:", xb.shape, "Batch y:", yb.shape)  # (B, T, C)

Batch X: (64, 300, 6) Batch y: (64,)


## LSTM model

In [25]:
from tensorflow import keras
from tensorflow.keras import layers

def build_bilstm(num_classes, T=300, C=6, lr=1e-3):
    inputs = keras.Input(shape=(T, C))
    x = layers.Masking(mask_value=0.0)(inputs)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Bidirectional(layers.LSTM(128))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_bilstm(num_classes=num_classes, T=T, C=C, lr=1e-3)
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 300, 6)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 300, 6)    │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_2 (Masking) │ (None, 300, 6)    │          0 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_2 (Any)         │ (None, 300)       │          0 │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 300, 256)  │    138,240 │ masking_2[0][0],  │
│ (Bidirectional)     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 300, 256)  │          0 │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 256)       │    394,240 │ dropout_5[0][0],  │
│ (Bidirectional)     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 256)       │          0 │ bidirectional_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 256)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 45)        │     11,565 │ dropout_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 609,837 (2.33 MB)

 Trainable params: 609,837 (2.33 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print("Test accuracy:", test_acc)

Epoch 1/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 45s 477ms/step - accuracy: 0.0338 - loss: 3.7618 - val_accuracy: 0.0722 - val_loss: 3.6306 - learning_rate: 0.0010
Epoch 2/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 56s 642ms/step - accuracy: 0.0802 - loss: 3.5082 - val_accuracy: 0.1079 - val_loss: 3.3850 - learning_rate: 0.0010
Epoch 3/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 56s 640ms/step - accuracy: 0.1071 - loss: 3.3604 - val_accuracy: 0.1187 - val_loss: 3.2703 - learning_rate: 0.0010
Epoch 4/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 55s 628ms/step - accuracy: 0.1265 - loss: 3.2623 - val_accuracy: 0.1494 - val_loss: 3.1661 - learning_rate: 0.0010
Epoch 5/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 57s 650ms/step - accuracy: 0.1441 - loss: 3.1751 - val_accuracy: 0.1651 - val_loss: 3.1168 - learning_rate: 0.0010
Epoch 6/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 57s 653ms/step - accuracy: 0.1598 - loss: 3.0891 - val_accuracy: 0.1859 - val_loss: 2.9817 - learning_rate: 0.0010
Epoch 7/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 58s 657ms/step - accuracy: 0.1772 - loss: 3.